# TrialMatch final held-out evaluation

Run once after freezing the prompt-v2 unweighted epoch-3 adapter. Select a CUDA GPU and upload the submission source ZIP and selected adapter ZIP. The two models use identical test prompts and settings.

## 1. Upload the updated project source ZIP

In [ ]:
from pathlib import Path
import hashlib
import os
import stat
import zipfile
from google.colab import files

uploaded = files.upload()
expected = 'trialmatch-colab-starter.zip'
if expected not in uploaded:
    raise ValueError(f'Upload the updated {expected}.')
bundle_bytes = uploaded[expected]
BUNDLE_SHA256 = hashlib.sha256(bundle_bytes).hexdigest()
PROJECT = Path('/content/trialmatch-training-' + BUNDLE_SHA256[:12])
archive_path = Path('/content') / expected
archive_path.write_bytes(bundle_bytes)
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    if not {'scripts/final_evaluation.py'} <= set(archive.namelist()):
        raise ValueError('This is an older bundle. Upload the updated ablation bundle.')
    for entry in archive.infolist():
        if not (PROJECT / entry.filename).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError('Archive contains an unsafe path')
        if stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError('Archive contains a symbolic link')
    archive.extractall(PROJECT)
del uploaded, bundle_bytes
os.chdir(PROJECT)
os.environ['HF_HOME'] = '/content/trialmatch-hf-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLCONFIGDIR'] = '/content/trialmatch-matplotlib'
print('Project:', PROJECT)
print('Bundle SHA256:', BUNDLE_SHA256)


## 2. Install and check GPU

The actual hardware and native BF16 support are recorded. A100 is required by this notebook
by default; set REQUIRE_A100=False explicitly if using another CUDA GPU. All implementations
also have CPU correctness tests and do not depend on a third-party CUDA attention kernel.


In [ ]:
import subprocess
import sys
import json

REQUIRE_A100 = False
# Use the uploaded project explicitly; preserve pip's actual error, not just its exit code.
if not (PROJECT / 'requirements-colab.txt').is_file() or not (PROJECT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Project setup files are missing. Rerun the updated source ZIP upload cell first.')
INSTALL_LOG = Path('/content/trialmatch-install.log')
command = [sys.executable, '-m', 'pip', 'install',
           '-r', str(PROJECT / 'requirements-colab.txt'), '-e', str(PROJECT)]
print('Installing project from:', PROJECT, flush=True)
with INSTALL_LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f'pip installation failed. The real error is above; full log: {INSTALL_LOG}. '
                       'Share that log or its final ERROR lines before changing package versions.')
hardware = json.loads(subprocess.check_output([sys.executable, '-c',
    'import torch,json; assert torch.cuda.is_available(), "Select a GPU runtime"; '
    'print(json.dumps({"gpu":torch.cuda.get_device_name(0),'
    '"memory_gib":torch.cuda.get_device_properties(0).total_memory/2**30,'
    '"torch":torch.__version__}))'], text=True))
print(hardware)
if REQUIRE_A100 and 'A100' not in hardware['gpu']:
    raise RuntimeError('This session is not using A100. Change the runtime GPU, or explicitly set REQUIRE_A100=False.')


## 3. Upload the selected adapter

Upload `trialmatch-selected-adapter.zip` supplied with this notebook. Its files are hashed
against the candidate manifest before model loading. No optimizer states or base weights
are included. Do not use the earlier one-step smoke adapter or the weighted B adapter.


In [ ]:
import hashlib
import zipfile
import stat

uploaded = files.upload()
name = 'trialmatch-selected-adapter.zip'
if name not in uploaded:
    raise ValueError('Upload ' + name)
archive_path = Path('/content') / name
archive_path.write_bytes(uploaded[name])
ADAPTER = Path('/content/trialmatch-selected-adapter')
ADAPTER.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    for entry in archive.infolist():
        if not (ADAPTER / entry.filename).resolve().is_relative_to(ADAPTER.resolve()):
            raise ValueError('Unsafe archive path')
        if stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError('Symbolic link in archive')
    archive.extractall(ADAPTER)
del uploaded
candidate = json.loads((PROJECT / 'configs/model_candidate.json').read_text())
for name, expected in candidate['adapter_file_sha256'].items():
    if hashlib.sha256((ADAPTER / name).read_bytes()).hexdigest() != expected:
        raise ValueError('Wrong adapter file: ' + name)
print('Selected adapter integrity passed.')


## Recreate the fixed split and evaluate

The output ZIP contains paired predictions and full metrics. Keep the selected model and prompt fixed.

In [ ]:
import shutil
manifest = json.loads((PROJECT / 'data/manifests/split_manifest.json').read_text())
subprocess.run([sys.executable, 'scripts/prepare_data.py', '--revision', manifest['dataset_revision']], cwd=PROJECT, check=True)
RUN_DIR = Path('/content/trialmatch-final-test')
subprocess.run([sys.executable, '-u', 'scripts/final_evaluation.py', '--adapter-dir', str(ADAPTER),
                '--output-dir', str(RUN_DIR), '--confirm-frozen'], cwd=PROJECT, check=True)
files.download(shutil.make_archive('/content/trialmatch-final-test', 'zip', root_dir=RUN_DIR))
